# Train ML Step Classifier

Fine-tune BERT/RoBERTa/DeBERTa on GSM8K + Math500 for step-level error classification.


In [ ]:
# Install dependencies
!pip install transformers datasets accelerate scikit-learn torch sympy streamlit


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent))

from src.data.loaders import prepare_training_data
import torch


In [ ]:
# Prepare data
print("Preparing training data...")
train_data, val_data, test_data = prepare_training_data(
    gsm8k_dir=".",
    math500_path="math_500_test.csv",
    output_dir="data/processed",
    seed=42
)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")


In [ ]:
# Run training script
!python scripts/train_classifier.py \
    --train_data data/processed/train.json \
    --val_data data/processed/val.json \
    --model_name roberta-base \
    --output_dir models/checkpoints/ \
    --batch_size 16 \
    --learning_rate 2e-5 \
    --num_epochs 5


In [ ]:
# Test inference
from src.models.ml_step_classifier import MLStepClassifierWrapper

classifier = MLStepClassifierWrapper(
    model_path="models/checkpoints/",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Test example
result = classifier.infer(
    problem="Natalia sold clips to 48 of her friends in April.",
    prev_steps="",
    current_step="Natalia sold 48/2 = 25 clips in May."
)

print(f"Label: {result['label']}")
print(f"Confidence: {result['confidence']:.3f}")
